# 13 · Informer (Transformer)

Zhou et al. (2021), *Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting* (AAAI).

Implementação compacta, fiel ao artigo e ao código oficial:
- **atenção ProbSparse**: só as u = c·ln(L) consultas mais "ativas" (medida M, estimada com uma amostra de chaves) fazem atenção completa. As demais recebem a média dos valores, ou a soma acumulada no decoder com máscara causal. Fator c = 5;
- **destilação** entre as camadas do encoder (convolução + BatchNorm + ELU + max-pooling de passo 2), que corta a sequência pela metade;
- **decoder generativo**: a entrada é a metade final da janela (`label_len = L/2`) seguida de H zeros, e o modelo prevê os H passos de uma vez;
- hiperparâmetros do artigo: `d_model=512`, 8 cabeças, 2 camadas no encoder e 1 no decoder, `d_ff=2048`, dropout 0.05.

**Diferenças em relação ao original:** sem embedding de tempo (o protocolo não usa covariáveis de calendário) e sem a opção `mix` do decoder.

**Custo:** é o maior modelo da comparação (11,3 milhões de parâmetros). Em GPU T4, cada execução levou de 2 a 14 min no ETTh1 e de 13 min a 1 h no Weather, conforme a janela.

In [1]:
# Configuração: igual em todos os notebooks
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJETO = '/content/drive/MyDrive/RN-Projeto'
except ImportError:          # fora do Colab (ex.: rodando no computador)
    PROJETO = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJETO, 'src'))
print('Pasta do projeto:', PROJETO)

import importlib
import rn_utils as ru
importlib.reload(ru)        # pega a versão mais nova se o 02 foi rodado de novo
ru.configurar(PROJETO)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/RN-Projeto
Projeto: /content/drive/MyDrive/RN-Projeto | dispositivo: cpu


## Modelo

`criar(L, H)` devolve um `nn.Module` que recebe `[lote, L, 1]` e devolve `[lote, H, 1]`. A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos os modelos.

In [2]:
import os
# reduz a fragmentação de memória da GPU (precisa ser definido antes do primeiro uso da GPU)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import math
import torch
import torch.nn as nn

class AtencaoProbSparse(nn.Module):
    """ProbSparse: só as consultas mais 'ativas' fazem atenção completa (Zhou et al., 2021)."""
    def __init__(self, mascara=False, fator=5):
        super().__init__()
        self.mascara, self.fator = mascara, fator

    def forward(self, q, k, v):                                  # [B, L, h, E]
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))         # [B, h, L, E]
        B, h, L_Q, E = q.shape
        L_K = k.shape[2]
        n_amostra = min(L_K, self.fator * math.ceil(math.log(L_K)))
        n_top = min(L_Q, self.fator * math.ceil(math.log(L_Q)))
        # 1) mede a "atividade" M de cada consulta com uma amostra de chaves
        idx = torch.randint(L_K, (L_Q, n_amostra), device=q.device)
        #    Só serve para escolher as consultas (topk não tem gradiente): sem grafo e em pedaços,
        #    para não montar de uma vez o tensor [B, h, L_Q, n_amostra, E] (17 GB na avaliação com L=512).
        with torch.no_grad():
            passo = max(1, 2**28 // (B * h * n_amostra * E))         # ~1 GB por pedaço
            qk_amostra = torch.cat([
                (q[:, :, i:i + passo].unsqueeze(-2) @ k[:, :, idx[i:i + passo], :].transpose(-2, -1)).squeeze(-2)
                for i in range(0, L_Q, passo)], dim=2)
            M = qk_amostra.max(-1)[0] - qk_amostra.sum(-1) / L_K
        top = M.topk(n_top, sorted=False)[1]                     # [B, h, n_top]
        # 2) atenção completa só para as consultas escolhidas
        q_top = torch.gather(q, 2, top[..., None].expand(-1, -1, -1, E))
        scores = q_top @ k.transpose(-2, -1) / math.sqrt(E)      # [B, h, n_top, L_K]
        if self.mascara:
            scores = scores.masked_fill(torch.arange(L_K, device=q.device) > top[..., None], float('-inf'))
        # 3) as outras consultas recebem a média (ou soma acumulada, com máscara) dos valores
        if self.mascara:
            contexto = v.cumsum(dim=2)
        else:
            contexto = v.mean(dim=2, keepdim=True).expand(-1, -1, L_Q, -1).clone()
        contexto = contexto.scatter(2, top[..., None].expand(-1, -1, -1, v.shape[-1]),
                                    torch.softmax(scores, dim=-1) @ v)
        return contexto.transpose(1, 2)                          # [B, L_Q, h, E]

class AtencaoCompleta(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v):                                  # [B, L, h, E]
        scores = torch.einsum('blhe,bshe->bhls', q, k) / math.sqrt(q.shape[-1])
        return torch.einsum('bhls,bshd->blhd', self.dropout(torch.softmax(scores, -1)), v)

class CamadaAtencao(nn.Module):
    def __init__(self, atencao, d_model, n_heads):
        super().__init__()
        self.atencao, self.h = atencao, n_heads
        self.q, self.k, self.v, self.o = (nn.Linear(d_model, d_model) for _ in range(4))

    def forward(self, q, k, v):
        B, Lq, _ = q.shape
        Lk = k.shape[1]
        out = self.atencao(self.q(q).view(B, Lq, self.h, -1), self.k(k).view(B, Lk, self.h, -1),
                           self.v(v).view(B, Lk, self.h, -1))
        return self.o(out.reshape(B, Lq, -1))

class Embedding(nn.Module):
    """Convolução sobre os valores + posição senoidal (sem embedding de tempo)."""
    def __init__(self, d_model, dropout, max_len=4096):
        super().__init__()
        self.token = nn.Conv1d(1, d_model, 3, padding=1, padding_mode='circular', bias=False)
        nn.init.kaiming_normal_(self.token.weight, mode='fan_in', nonlinearity='leaky_relu')
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):                                        # [B, L, 1]
        return self.dropout(self.token(x.transpose(1, 2)).transpose(1, 2) + self.pe[:x.shape[1]])

def _ffn(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

class CamadaEncoder(nn.Module):
    def __init__(self, atencao, d_model, d_ff, dropout):
        super().__init__()
        self.atencao, self.ff = atencao, _ffn(d_model, d_ff, dropout)
        self.n1, self.n2, self.drop = nn.LayerNorm(d_model), nn.LayerNorm(d_model), nn.Dropout(dropout)

    def forward(self, x):
        x = self.n1(x + self.drop(self.atencao(x, x, x)))
        return self.n2(x + self.drop(self.ff(x)))

class Destilacao(nn.Module):
    """Conv + BatchNorm + ELU + MaxPool(passo 2): reduz a sequência à metade."""
    def __init__(self, d_model):
        super().__init__()
        self.rede = nn.Sequential(nn.Conv1d(d_model, d_model, 3, padding=1, padding_mode='circular'),
                                  nn.BatchNorm1d(d_model), nn.ELU(), nn.MaxPool1d(3, stride=2, padding=1))

    def forward(self, x):
        return self.rede(x.transpose(1, 2)).transpose(1, 2)

class CamadaDecoder(nn.Module):
    def __init__(self, auto, cruzada, d_model, d_ff, dropout):
        super().__init__()
        self.auto, self.cruzada, self.ff = auto, cruzada, _ffn(d_model, d_ff, dropout)
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d_model) for _ in range(3))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memoria):
        x = self.n1(x + self.drop(self.auto(x, x, x)))
        x = self.n2(x + self.drop(self.cruzada(x, memoria, memoria)))
        return self.n3(x + self.drop(self.ff(x)))

class Informer(nn.Module):
    def __init__(self, L, H, d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, fator=5, dropout=0.05):
        super().__init__()
        self.H, self.label_len = H, L // 2
        self.emb_enc, self.emb_dec = Embedding(d_model, dropout), Embedding(d_model, dropout)
        self.encoder = nn.ModuleList([
            CamadaEncoder(CamadaAtencao(AtencaoProbSparse(False, fator), d_model, n_heads), d_model, d_ff, dropout)
            for _ in range(e_layers)])
        self.destilacao = nn.ModuleList([Destilacao(d_model) for _ in range(e_layers - 1)])
        self.norm_enc = nn.LayerNorm(d_model)
        self.decoder = nn.ModuleList([
            CamadaDecoder(CamadaAtencao(AtencaoProbSparse(True, fator), d_model, n_heads),
                          CamadaAtencao(AtencaoCompleta(dropout), d_model, n_heads), d_model, d_ff, dropout)
            for _ in range(d_layers)])
        self.norm_dec = nn.LayerNorm(d_model)
        self.saida = nn.Linear(d_model, 1)

    def forward(self, x):                                        # [B, L, 1] → [B, H, 1]
        memoria = self.emb_enc(x)
        for i, camada in enumerate(self.encoder):
            memoria = camada(memoria)
            if i < len(self.destilacao):
                memoria = self.destilacao[i](memoria)
        memoria = self.norm_enc(memoria)
        x_dec = torch.cat([x[:, -self.label_len:], torch.zeros_like(x[:, :self.H])], dim=1)
        h = self.emb_dec(x_dec)
        for camada in self.decoder:
            h = camada(h, memoria)
        return self.saida(self.norm_dec(h))[:, -self.H:]

def criar(L, H):
    return Informer(L, H)

In [3]:
# Verificação do formato de entrada e saída
import torch
for L in (96, 720):
    m = criar(L, ru.HORIZONTE)
    saida = m(torch.randn(4, L, 1))
    assert saida.shape == (4, ru.HORIZONTE, 1), saida.shape
    print(f'L={L}: saída {tuple(saida.shape)} ✓ | {sum(p.numel() for p in m.parameters()):,} parâmetros')

L=96: saída (4, 96, 1) ✓ | 11,302,401 parâmetros
L=720: saída (4, 96, 1) ✓ | 11,302,401 parâmetros


## Treino e avaliação

O learning rate é escolhido pela validação (L = 336, seed 0) entre {1e-3, 3e-4, 1e-4}. Depois o modelo é treinado em todas as janelas e seeds das duas bases e avaliado no teste, com a janela original e com as quatro janelas embaralhadas. As execuções já salvas em `resultados/` são apenas recarregadas e listadas.

In [4]:
ru.rodar_grade('Informer', criar)

lr escolhido para Informer em ETTh1: 0.0001  (val: {'0.001': 0.1125, '0.0003': 0.1546, '0.0001': 0.1081})
  · ETTh1_Informer_L96_H96_lr0.0001_s0 (salvo) | val 0.1192 | teste MSE 0.0623 MAE 0.1938 | embaralhar_tudo ×1.89 | 6 épocas, 138s (cuda)
  · ETTh1_Informer_L96_H96_lr0.0001_s1 (salvo) | val 0.1144 | teste MSE 0.0622 MAE 0.1955 | embaralhar_tudo ×1.61 | 6 épocas, 138s (cuda)
  · ETTh1_Informer_L96_H96_lr0.0001_s2 (salvo) | val 0.1180 | teste MSE 0.0646 MAE 0.1996 | embaralhar_tudo ×1.88 | 7 épocas, 161s (cuda)
  · ETTh1_Informer_L192_H96_lr0.0001_s0 (salvo) | val 0.1121 | teste MSE 0.0587 MAE 0.1848 | embaralhar_tudo ×2.64 | 6 épocas, 231s (cuda)
  · ETTh1_Informer_L192_H96_lr0.0001_s1 (salvo) | val 0.1156 | teste MSE 0.0807 MAE 0.2332 | embaralhar_tudo ×2.17 | 6 épocas, 231s (cuda)
  · ETTh1_Informer_L192_H96_lr0.0001_s2 (salvo) | val 0.1212 | teste MSE 0.0641 MAE 0.2005 | embaralhar_tudo ×1.86 | 7 épocas, 269s (cuda)
  · ETTh1_Informer_L336_H96_lr0.0001_s0 (salvo) | val 0.1081 | 

## Resumo (média das 3 seeds)

`×tudo`, `×antigo`, `×recente` e `×blocos` = MSE com a janela embaralhada / MSE original. ×1,00 significa que o modelo ignora aquela ordem.

In [5]:
df = ru.carregar_resultados()
r = df[df.modelo == 'Informer']
resumo = r.groupby(['dataset', 'L']).agg(**{
    'MSE val': ('mse_val', 'mean'), 'MSE teste': ('original_mse', 'mean'), 'dp': ('original_mse', 'std'),
    'MAE teste': ('original_mae', 'mean'), 'MSE sem suspeitos': ('original_mse_ss', 'mean'),
    **{'×' + p.replace('embaralhar_', ''): (f'{p}_razao', 'mean') for p in ru.PERTURBACOES},
    'épocas': ('epocas_rodadas', 'mean'), 'tempo (s)': ('tempo_treino_s', 'mean')})
resumo.round(4)

MSE val  MSE teste      dp  MAE teste  MSE sem suspeitos  \
dataset L                                                               
ETTh1   96    0.1172     0.0630  0.0014     0.1963             0.0617   
        192   0.1163     0.0678  0.0115     0.2062             0.0692   
        336   0.1126     0.0688  0.0055     0.2059             0.0717   
        512   0.1176     0.0761  0.0036     0.2160             0.0799   
        720   0.1484     0.0797  0.0071     0.2212             0.0834   
Weather 96    0.5573     0.4951  0.0141     0.4869             0.4951   
        336   0.5666     0.5355  0.0261     0.5007             0.5355   
        720   0.5625     0.4825  0.0193     0.4932             0.4825   

               ×tudo  ×antigo  ×recente  ×blocos  épocas  tempo (s)  
dataset L                                                            
ETTh1   96    1.7931   1.4013    1.1103   1.1821  6.3333   145.8441  
        192   2.2210   1.3614    1.1204   1.2988  6.3333   243.9082  
        336   1.5716   1.0274    1.0646   1.2043  6.0000   373.4080  
        512   2.1840   1.6780    1.0920   1.2087  7.0000   609.9246  
        720   1.3918   1.0627    1.2322   1.2210  6.3333   768.9538  
Weather 96    1.2581   1.0764    1.0642   1.2846  8.3333   869.3002  
        336   6.8624   2.8243    2.4360   1.4749  8.3333  2312.7446  
        720  16.8345   5.7677    2.7540   1.5214  6.0000  3355.4301